# 05 — Ablation Study

On isole **chaque choix de design** pour mesurer son impact individuel.
C'est ce qu'on appelle une *ablation study* en ML : on retire ou fait varier
un composant à la fois en gardant tout le reste fixe.

**Axes étudiés :**
1. **Nombre de qubits** → dimension PCA, taille de l'espace quantique
2. **Bandwidth α** → paramètre de largeur du kernel
3. **Nombre de kernels** → combien en faut-il vraiment ?
4. **Type de kernel** → Fidelity vs Projected
5. **Concentration exponentielle** → évolution avec n_qubits

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

from src.preprocessing import QuantumScaler, FeatureReducer
from src.kernels import build_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.kernels.kernel_matrix import kernel_statistics
from src.kernels.feature_maps import get_feature_map_library
from src.mkl.alignment import centered_alignment
from src.evaluation.ablation import ablation_n_kernels, weight_analysis

import os
os.makedirs('../results', exist_ok=True)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

In [ ]:
# Dataset de référence
N_SAMPLES = 200

X_raw, y = make_classification(
    n_samples=N_SAMPLES, n_features=20, n_informative=12,
    n_redundant=5, random_state=SEED, weights=[0.7, 0.3]
)
print(f'Dataset: {X_raw.shape}, Classes: {np.bincount(y)}')

## Ablation 1 : Impact du nombre de qubits

**Question** : est-ce que plus de qubits = meilleure performance ?

**Attendu** : performances stables ou légèrement croissantes jusqu'à un plateau,
puis chute due à la concentration exponentielle (si sans MKL).

In [ ]:
QUBIT_RANGE = [2, 4, 6, 8]  # ↑ = plus lent ; IBM teste jusqu'à 20
N_RUNS_ABL  = 5

def run_single_qubit_experiment(X_raw, y, n_qubits, method='mkl', n_runs=5):
    """Entraîne et évalue QMKL pour un n_qubits donné."""
    reducer = FeatureReducer(n_components=n_qubits)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X = scaler.fit_transform(reducer.fit_transform(X_raw))

    fm_lib = get_feature_map_library(n_qubits)  # returns OrderedDict
    K_full = []
    for label, fm in fm_lib.items():
        qk = build_quantum_kernel(fm, kernel_type='fidelity')
        K_full.append(compute_kernel_matrix(qk, X))

    # Statistiques de concentration
    conc_stats = [kernel_statistics(K) for K in K_full]
    mean_of_means = np.mean([s['mean'] for s in conc_stats])
    mean_of_vars  = np.mean([s['variance'] for s in conc_stats])

    scores = []
    for seed in range(n_runs):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_full]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_full]

        if method == 'mkl':
            w = centered_alignment(K_tr, np.outer(y_tr, y_tr).astype(float))
        else:  # single best
            Ky = np.outer(y_tr, y_tr).astype(float)
            aligns = [np.sum(K * Ky) / (np.linalg.norm(K,'fro') * np.linalg.norm(Ky,'fro') + 1e-10)
                      for K in K_tr]
            w = np.zeros(len(K_tr)); w[np.argmax(aligns)] = 1.0

        w = np.array(w); w = w / (w.sum() + 1e-10)
        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))

        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])

        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    return {
        'mean': np.mean(scores), 'std': np.std(scores, ddof=1),
        'concentration_mean': mean_of_means,
        'concentration_var':  mean_of_vars,
    }

print('Testing n_qubits...')
abl_qubits_mkl    = {}
abl_qubits_single = {}

for nq in QUBIT_RANGE:
    print(f'  n_qubits={nq}')
    abl_qubits_mkl[nq]    = run_single_qubit_experiment(X_raw, y, nq, 'mkl',    N_RUNS_ABL)
    abl_qubits_single[nq] = run_single_qubit_experiment(X_raw, y, nq, 'single', N_RUNS_ABL)

print('Done!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dims  = QUBIT_RANGE
mkl_m = [abl_qubits_mkl[d]['mean']    for d in dims]
mkl_s = [abl_qubits_mkl[d]['std']     for d in dims]
sng_m = [abl_qubits_single[d]['mean'] for d in dims]
sng_s = [abl_qubits_single[d]['std']  for d in dims]

# ROC-AUC vs n_qubits
ax = axes[0]
ax.errorbar(dims, mkl_m, yerr=mkl_s, fmt='o-', label='QMKL-Centered', color='#2ecc71', capsize=5, linewidth=2)
ax.errorbar(dims, sng_m, yerr=sng_s, fmt='s--', label='Single-Best',   color='#e74c3c', capsize=5, linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('ROC-AUC')
ax.set_title('Performance vs n_qubits')
ax.legend()

# Concentration : mean
ax = axes[1]
conc_means = [abl_qubits_mkl[d]['concentration_mean'] for d in dims]
ax.plot(dims, conc_means, 'o-', color='#3498db', linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('Moyenne hors-diagonale du kernel')
ax.set_title('Concentration exponentielle (moyenne)')
ax.text(0.05, 0.05, '→ Tend vers 0\n  = concentration',
        transform=ax.transAxes, fontsize=9, color='red', alpha=0.7)

# Concentration : variance
ax = axes[2]
conc_vars = [abl_qubits_mkl[d]['concentration_var'] for d in dims]
ax.plot(dims, conc_vars, 's-', color='#9b59b6', linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('Variance hors-diagonale du kernel')
ax.set_title('Concentration exponentielle (variance)')
ax.text(0.05, 0.05, '→ Tend vers 0\n  = perd discrimination',
        transform=ax.transAxes, fontsize=9, color='red', alpha=0.7)

plt.suptitle('Ablation — Impact du nombre de qubits', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_qubits.png', dpi=150, bbox_inches='tight')
plt.show()

## Ablation 2 : Impact du bandwidth α

**Question** : quelle valeur d'α donne les meilleures performances ?

**Attendu** : courbe en cloche — trop petit = pas de discrimination, trop grand = concentration.

In [ ]:
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

ALPHA_RANGE = [0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 14.0, 20.0]
FM_NAMES    = ['Z', 'ZZ']

abl_alpha = {fm: {} for fm in FM_NAMES}

for fm_name in FM_NAMES:
    print(f'\nFeature map: {fm_name}')
    for alpha in ALPHA_RANGE:
        fm = build_feature_map(fm_name, N_QUBITS_FIXED, alpha=alpha, reps=1)
        qk = build_quantum_kernel(fm, kernel_type='fidelity')
        K_full = compute_kernel_matrix(qk, X_fixed)
        stats  = kernel_statistics(K_full)

        scores = []
        for seed in range(5):
            idx = np.arange(len(y))
            idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                              random_state=seed, stratify=y)
            K_tr = K_full[np.ix_(idx_tr, idx_tr)]
            K_te = K_full[np.ix_(idx_te, idx_tr)]
            ev   = np.linalg.eigvalsh(K_tr)
            if ev.min() < 0:
                K_tr += (abs(ev.min()) + 1e-10) * np.eye(K_tr.shape[0])
            svm = SVC(kernel='precomputed', C=1.0, probability=True)
            svm.fit(K_tr, y[idx_tr])
            scores.append(roc_auc_score(y[idx_te], svm.predict_proba(K_te)[:,1]))

        abl_alpha[fm_name][alpha] = {
            'mean': np.mean(scores), 'std': np.std(scores, ddof=1),
            'k_mean': stats['mean'], 'k_var': stats['variance']
        }
        print(f'  α={alpha:5.1f}: AUC={np.mean(scores):.4f}, K_mean={stats["mean"]:.4f}, K_var={stats["variance"]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Z': '#2ecc71', 'ZZ': '#e67e22'}

for ax_idx, (metric, ylabel, title) in enumerate([
    ('auc', 'ROC-AUC', 'Performance vs α'),
    ('kvar', 'Variance hors-diagonale', 'Concentration du kernel vs α'),
]):
    ax = axes[ax_idx]
    for fm_name in FM_NAMES:
        alphas = sorted(abl_alpha[fm_name].keys())
        if metric == 'auc':
            vals = [abl_alpha[fm_name][a]['mean'] for a in alphas]
            errs = [abl_alpha[fm_name][a]['std']  for a in alphas]
            ax.errorbar(alphas, vals, yerr=errs, fmt='o-',
                        label=fm_name, color=colors[fm_name], capsize=4, linewidth=2)
        else:
            vals = [abl_alpha[fm_name][a]['k_var'] for a in alphas]
            ax.plot(alphas, vals, 'o-', label=fm_name, color=colors[fm_name], linewidth=2)
    ax.set_xlabel('Bandwidth α', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.set_xscale('log')

plt.suptitle('Ablation — Impact du bandwidth α', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

## Ablation 3 : Combien de kernels faut-il ?

**Question** : est-ce que plus de kernels = toujours mieux ?

**Méthode améliorée** :
- **Ranking par consensus** : alignement moyenné sur 5 folds (pas 1 seul seed)
- **K-fold CV stratifié** : 3 rounds × 5 folds = 15 évaluations par n_k → variance stable
- **Intervalle de confiance à 95%** : bande grisée au lieu de barres d'erreur

In [ ]:
# ── Préparation des données et kernels ────────────────────────────────────────
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

fm_lib = get_feature_map_library(N_QUBITS_FIXED)  # OrderedDict
knames = list(fm_lib.keys())
K_full = []
for label, fm in fm_lib.items():
    qk = build_quantum_kernel(fm, kernel_type='fidelity')
    K_full.append(compute_kernel_matrix(qk, X_fixed))

print(f'{len(K_full)} kernel matrices computed.\n')

# ── Ablation n_kernels (K-fold CV + consensus ranking) ────────────────────────
def mkl_weight_fn(K_list_train, y_train):
    K_target = np.outer(y_train, y_train).astype(float)
    return centered_alignment(K_list_train, K_target)

nk_results, ranking_info = ablation_n_kernels(
    K_full, y,
    weight_fn=mkl_weight_fn,
    kernel_names=knames,
    n_folds=5,
    C=1.0,
    scoring='roc_auc',
    random_state=SEED,
)

print(f'\nConsensus ranking: {ranking_info["ranked_names"]}')

In [ ]:
# ── Visualisation avec bande de confiance 95% ─────────────────────────────────
n_ks   = sorted(nk_results.keys())
means  = [nk_results[k]['mean'] for k in n_ks]
ci95s  = [nk_results[k]['ci95'] for k in n_ks]
added  = [nk_results[k]['added_kernel'] for k in n_ks]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel 1 : Courbe AUC vs n_kernels avec CI95 ──────────────────────────────
ax = axes[0]
ax.plot(n_ks, means, 'o-', color='#2980b9', linewidth=2.5, markersize=8, zorder=3)
ax.fill_between(n_ks,
                [m - c for m, c in zip(means, ci95s)],
                [m + c for m, c in zip(means, ci95s)],
                alpha=0.2, color='#2980b9', label='CI 95%')

# Annoter le kernel ajouté
for i, (nk, m, name) in enumerate(zip(n_ks, means, added)):
    short = name[:12]
    ax.annotate(f'+{short}', (nk, m), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=7, color='#2c3e50',
                rotation=30)

# Marquer le maximum
best_k = n_ks[np.argmax(means)]
ax.axvline(best_k, color='#e74c3c', linestyle='--', alpha=0.7,
           label=f'Optimal: {best_k} kernels (AUC={max(means):.3f})')

ax.set_xlabel('Nombre de kernels combinés', fontsize=12)
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('Performance vs nombre de kernels\n(3 rounds × 5-fold CV = 15 évals/point)', fontsize=11)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xticks(n_ks)

# ── Panel 2 : Poids moyens des kernels à l'optimum ───────────────────────────
ax = axes[1]
opt_weights = nk_results[best_k]['mean_weights']
opt_names   = ranking_info['ranked_names'][:best_k]
colors = ['#2ecc71' if w > 0.05 else '#e74c3c' for w in opt_weights]
bars = ax.barh(range(best_k), opt_weights, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(best_k))
ax.set_yticklabels(opt_names, fontsize=9)
ax.set_xlabel('Poids moyen', fontsize=11)
ax.set_title(f'Poids des {best_k} kernels sélectionnés\n(vert = actif, rouge = quasi-nul)', fontsize=11)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Ablation — Impact du nombre de kernels', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/05_ablation_n_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n→ Optimal : {best_k} kernels (AUC = {max(means):.4f})')
print(f'  Kernels actifs : {[n for n, w in zip(opt_names, opt_weights) if w > 0.05]}')
print(f'  Kernels quasi-nuls : {[n for n, w in zip(opt_names, opt_weights) if w <= 0.05]}')

In [ ]:
# ── Analyse des poids : POURQUOI certains kernels ont un poids nul ? ──────────
analysis, W_matrix = weight_analysis(
    K_full, y,
    weight_fn=mkl_weight_fn,
    kernel_names=knames,
    n_folds=5, n_rounds=3, random_state=SEED,
)

# Table résumé
df = pd.DataFrame(analysis).T
df = df.sort_values('mean_weight', ascending=False)
print('=== Analyse des poids par kernel ===')
print(df[['mean_weight', 'nonzero_rate', 'alignment', 'off_diag_std']].to_string(float_format='%.4f'))

# Scatter : alignment vs poids moyen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, row in df.iterrows():
    color = '#2ecc71' if row['nonzero_rate'] > 0.5 else '#e74c3c'
    ax.scatter(row['alignment'], row['mean_weight'], color=color, s=100, edgecolors='black', zorder=3)
    ax.annotate(name[:10], (row['alignment'], row['mean_weight']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Alignement kernel-target', fontsize=11)
ax.set_ylabel('Poids MKL moyen', fontsize=11)
ax.set_title('Alignement vs Poids MKL\n(vert = souvent actif, rouge = souvent nul)')
ax.grid(True, alpha=0.3)

# Scatter : concentration (off-diag std) vs poids moyen
ax = axes[1]
for name, row in df.iterrows():
    color = '#2ecc71' if row['nonzero_rate'] > 0.5 else '#e74c3c'
    ax.scatter(row['off_diag_std'], row['mean_weight'], color=color, s=100, edgecolors='black', zorder=3)
    ax.annotate(name[:10], (row['off_diag_std'], row['mean_weight']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Écart-type hors-diagonale (pouvoir discriminant)', fontsize=11)
ax.set_ylabel('Poids MKL moyen', fontsize=11)
ax.set_title('Concentration vs Poids MKL\n(faible std = concentré = inutile)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/05_weight_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Conclusion automatique
nul = [n for n, r in analysis.items() if r['nonzero_rate'] < 0.5]
actif = [n for n, r in analysis.items() if r['nonzero_rate'] >= 0.5]
print(f'\n→ Kernels actifs ({len(actif)}/{len(knames)}): {actif}')
print(f'→ Kernels quasi-nuls ({len(nul)}/{len(knames)}): {nul}')
print('  Raisons probables : faible alignement OU forte concentration (std ≈ 0)')

N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

kernel_types = ['fidelity', 'projected']
ktype_results = {}

for ktype in kernel_types:
    print(f'\nKernel type: {ktype}')
    fm_lib = get_feature_map_library(N_QUBITS_FIXED)  # OrderedDict
    K_full_list = []
    conc_stats  = []

    for label, fm in fm_lib.items():
        try:
            qk = build_quantum_kernel(fm, kernel_type=ktype, gamma=1.0)
            K  = compute_kernel_matrix(qk, X_fixed)
            K_full_list.append(K)
            conc_stats.append(kernel_statistics(K))
        except Exception as e:
            print(f'  Skipped {label}: {e}')

    scores = []
    for seed in range(5):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_full_list]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_full_list]
        w = centered_alignment(K_tr, np.outer(y_tr, y_tr).astype(float))
        w = np.array(w); w = w / (w.sum() + 1e-10)
        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    ktype_results[ktype] = {
        'scores': scores,
        'mean': np.mean(scores),
        'std': np.std(scores, ddof=1),
        'conc_mean': np.mean([s['mean'] for s in conc_stats]),
        'conc_var':  np.mean([s['variance'] for s in conc_stats]),
    }
    print(f'  AUC: {np.mean(scores):.4f} ± {np.std(scores,ddof=1):.4f}')
    print(f'  Kernel mean: {ktype_results[ktype]["conc_mean"]:.4f}')
    print(f'  Kernel var:  {ktype_results[ktype]["conc_var"]:.6f}')

In [ ]:
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

kernel_types = ['fidelity', 'projected']
ktype_results = {}

for ktype in kernel_types:
    print(f'\nKernel type: {ktype}')
    fm_lib = get_feature_map_library(N_QUBITS_FIXED)
    K_full_list = []
    conc_stats  = []

    for label, fm in fm_lib:
        try:
            qk = build_quantum_kernel(fm, kernel_type=ktype, gamma=1.0)
            K  = compute_kernel_matrix(qk, X_fixed)
            K_full_list.append(K)
            conc_stats.append(kernel_statistics(K))
        except Exception as e:
            print(f'  Skipped {label}: {e}')

    scores = []
    for seed in range(5):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_full_list]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_full_list]
        w = centered_alignment(K_tr, np.outer(y_tr, y_tr).astype(float))
        w = np.array(w); w = w / (w.sum() + 1e-10)
        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    ktype_results[ktype] = {
        'scores': scores,
        'mean': np.mean(scores),
        'std': np.std(scores, ddof=1),
        'conc_mean': np.mean([s['mean'] for s in conc_stats]),
        'conc_var':  np.mean([s['variance'] for s in conc_stats]),
    }
    print(f'  AUC: {np.mean(scores):.4f} ± {np.std(scores,ddof=1):.4f}')
    print(f'  Kernel mean: {ktype_results[ktype]["conc_mean"]:.4f}')
    print(f'  Kernel var:  {ktype_results[ktype]["conc_var"]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'fidelity': '#3498db', 'projected': '#e67e22'}

# Scores distribution
ax = axes[0]
for ktype, data in ktype_results.items():
    ax.boxplot(data['scores'], positions=[list(ktype_results.keys()).index(ktype)],
               widths=0.4, patch_artist=True,
               boxprops=dict(facecolor=colors[ktype], alpha=0.6))
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('ROC-AUC')
ax.set_title('Distribution des scores')

# Concentration mean
ax = axes[1]
for i, (ktype, data) in enumerate(ktype_results.items()):
    ax.bar(i, data['conc_mean'], color=colors[ktype], alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('Moyenne du kernel')
ax.set_title('Concentration moyenne')

# Concentration var
ax = axes[2]
for i, (ktype, data) in enumerate(ktype_results.items()):
    ax.bar(i, data['conc_var'], color=colors[ktype], alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('Variance du kernel')
ax.set_title('Concentration variance\n(↑ = plus discriminant)')

plt.suptitle('Ablation — Fidelity vs Projected Quantum Kernel', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_kernel_type.png', dpi=150, bbox_inches='tight')
plt.show()